In [ ]:
import pandas as pd
import numpy as np

import torch
import torch.nn as nn
import torch.optim as optim

import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import r2_score

from torch.utils.data import TensorDataset, DataLoader

In [ ]:
# Load dataset

df = pd.read_csv("powerplant_data.csv")

df.head()

# Check missing values

df.isnull().sum()

In [ ]:
X = df.drop("PE", axis=1)
y = df["PE"]

y.head()

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

# Separate validation data

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.2,
    random_state=42
)

In [ ]:
scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

In [ ]:
X_train_tensor = torch.tensor(
    X_train_scaled,
    dtype=torch.float32
)

X_val_tensor = torch.tensor(
    X_val_scaled,
    dtype=torch.float32
)

X_test_tensor = torch.tensor(
    X_test_scaled,
    dtype=torch.float32
)

y_train_tensor = torch.tensor(
    y_train.values,
    dtype=torch.float32
).view(-1, 1)

y_val_tensor = torch.tensor(
    y_val.values,
    dtype=torch.float32
).view(-1, 1)

y_test_tensor = torch.tensor(
    y_test.values,
    dtype=torch.float32
).view(-1, 1)

In [ ]:
train_dataset = TensorDataset(
    X_train_tensor,
    y_train_tensor
)

val_dataset = TensorDataset(
    X_val_tensor,
    y_val_tensor
)

test_dataset = TensorDataset(
    X_test_tensor,
    y_test_tensor
)

In [ ]:
train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True
)

val_loader = DataLoader(
    val_dataset,
    batch_size=32
)

test_loader = DataLoader(
    test_dataset,
    batch_size=32
)

In [ ]:
class ANN(nn.Module):

    def __init__(self):
        super().__init__()

        self.model = nn.Sequential(

            # 1st Hidden Layer
            nn.Linear(X_train.shape[1], 6),
            nn.ReLU(),

            # 2nd Hidden Layer
            nn.Linear(6, 6),
            nn.ReLU(),

            # Output Layer
            nn.Linear(6, 1)

        )

    def forward(self, x):

        return self.model(x)

In [ ]:
model = ANN()

# Loss Function

criterion = nn.MSELoss()

# Optimizer

optimizer = optim.Adam(model.parameters())

In [ ]:
train_losses = []
val_losses = []

epochs = 100

best_val_loss = float("inf")

for epoch in range(epochs):

    model.train()

    running_loss = 0.0

    for xb, yb in train_loader:

        # Reset gradients
        optimizer.zero_grad()

        # Forward propagation
        outputs = model(xb)

        # Calculate loss
        loss = criterion(outputs, yb)

        # Backward propagation
        loss.backward()

        # Update parameters
        optimizer.step()

        running_loss += loss.item() * xb.size(0)

    epoch_train_loss = running_loss / len(train_loader.dataset)

    train_losses.append(epoch_train_loss)


    # Validation

    running_val_loss = 0.0

    model.eval()

    with torch.no_grad():

        for xb, yb in val_loader:

            outputs = model(xb)

            loss = criterion(outputs, yb)

            running_val_loss += loss.item() * xb.size(0)

    epoch_val_loss = running_val_loss / len(val_loader.dataset)

    val_losses.append(epoch_val_loss)


    # Save Best Model

    if epoch_val_loss < best_val_loss:

        best_val_loss = epoch_val_loss

        torch.save(
            model.state_dict(),
            "best_model.pth"
        )


    print(
        f"Epoch {epoch+1}/{epochs} "
        f"==> Train Loss = {epoch_train_loss:.4f} "
        f"& Val Loss = {epoch_val_loss:.4f}"
    )

In [ ]:
loss_df = pd.DataFrame({

    "Training Loss": train_losses,

    "Validation Loss": val_losses

})

plt.plot(
    loss_df["Training Loss"],
    label="Training Loss"
)

plt.plot(
    loss_df["Validation Loss"],
    label="Validation Loss"
)

plt.xlabel("Epochs")

plt.ylabel("Losses")

plt.legend()

plt.show()

In [ ]:
model.load_state_dict(
    torch.load("best_model.pth")
)

In [ ]:
model.eval()

with torch.no_grad():

    train_preds = model(X_train_tensor)

    test_preds = model(X_test_tensor)

In [ ]:
train_mse_loss = criterion(
    train_preds,
    y_train_tensor
)

test_mse_loss = criterion(
    test_preds,
    y_test_tensor
)

print("Training MSE:", train_mse_loss.item())

print("Testing MSE:", test_mse_loss.item())

In [ ]:
from sklearn.metrics import r2_score

print(
    "R2 Score =",
    r2_score(
        y_test.values,
        test_preds.numpy().flatten()
    )
)

In [ ]:
predicted_df = pd.DataFrame(
    test_preds.numpy(),
    columns=["Predicted Values"]
)

actual_df = pd.DataFrame(
    y_test.values,
    columns=["Actual Values"]
)

pd.concat(
    [predicted_df, actual_df],
    axis=1
)